# backward-func-lookup — faded example 3: Complete the per-edge get_back_func dispatch

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `backward-func-lookup`. The last cell reports your progress on the `Backprop: BackwardFuncLookup` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: BackwardFuncLookup` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`backward-func-lookup`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "backward-func-lookup"
DD_SUBTOPIC = "Backprop: BackwardFuncLookup"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

In a reverse pass, for each parent edge of a node you look up the back fn by `(node_func, argnum)` and apply it. For a binary op the lookup is queried once per argnum. The dispatched gradients must match autograd.

## Faded exercise 3

`edge_grads(BFL, func, out, x, y, grad_out)` loops over argnums 0 and 1, looks up the back fn for `(func, argnum)`, and applies it. Complete the single line that retrieves the back fn for the current argnum.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
t.manual_seed(0)

class BackwardFuncLookup:
    def __init__(self):
        self.back_funcs = {}
    def add_back_func(self, forward_fn, argnum, back_fn):
        self.back_funcs[(forward_fn, argnum)] = back_fn
    def get_back_func(self, forward_fn, argnum):
        key = (forward_fn, argnum)
        if key not in self.back_funcs:
            raise KeyError(f'No back_fn for ({forward_fn!r}, argnum={argnum}).')
        return self.back_funcs[key]

def mul_back0(grad_out, out, x, y):
    return grad_out * y
def mul_back1(grad_out, out, x, y):
    return grad_out * x

def edge_grads(BFL, func, out, x, y, grad_out):
    grads = {}
    for argnum in (0, 1):
        back_fn = None  # TODO: fill in this step — read the prompt cell above
        grads[argnum] = back_fn(grad_out, out, x, y)
    return grads


def _test():
    BFL = BackwardFuncLookup()
    BFL.add_back_func(t.multiply, 0, mul_back0)
    BFL.add_back_func(t.multiply, 1, mul_back1)
    x = t.tensor([2.0, 3.0, 4.0])
    y = t.tensor([5.0, 6.0, 7.0])
    out = x * y
    grad_out = t.ones_like(out)
    grads = edge_grads(BFL, t.multiply, out, x, y, grad_out)
    # independent autograd reference
    xr = x.clone().detach().requires_grad_(True)
    yr = y.clone().detach().requires_grad_(True)
    (xr * yr).sum().backward()
    assert t.allclose(grads[0], xr.grad)
    assert t.allclose(grads[1], yr.grad)


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
t.manual_seed(0)

class BackwardFuncLookup:
    def __init__(self):
        self.back_funcs = {}
    def add_back_func(self, forward_fn, argnum, back_fn):
        self.back_funcs[(forward_fn, argnum)] = back_fn
    def get_back_func(self, forward_fn, argnum):
        key = (forward_fn, argnum)
        if key not in self.back_funcs:
            raise KeyError(f'No back_fn for ({forward_fn!r}, argnum={argnum}).')
        return self.back_funcs[key]

def mul_back0(grad_out, out, x, y):
    return grad_out * y
def mul_back1(grad_out, out, x, y):
    return grad_out * x

def edge_grads(BFL, func, out, x, y, grad_out):
    grads = {}
    for argnum in (0, 1):
        back_fn = BFL.get_back_func(func, argnum)
        grads[argnum] = back_fn(grad_out, out, x, y)
    return grads
```
</details>